# 15 — Análise Exploratória de Dados (EDA)
**Credit Risk Intelligence Platform** — EDA Completo e Auditável

Este notebook realiza análise exploratória completa dos datasets ML (`ml_train`, `ml_test`, `ml_feature_metadata`), cobrindo: distribuição do TARGET, qualidade dos dados, comportamento das features, correlações, importância preliminar, drift Train×Test e insights de negócio.

## Princípios

> **Notebook exploratório** — não altera datasets, não cria features permanentes, não aplica SMOTE, não treina modelos finais.
> Performance: evita `collect()` excessivo, usa amostragem para visualizações pesadas, agrega em batch quando possível.

In [0]:
# ============================================================================
# CÉLULA 1 — Imports, Configurações e Constantes
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
    DoubleType, LongType, BooleanType, TimestampType)
from datetime import datetime, timezone
import uuid
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# ----------------------------------------------------------------------------
# Identificadores de execução
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "eda_v1.0"
NOTEBOOK_NAME = "15_eda"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"eda_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)
EXEC_START = EXECUTION_TIMESTAMP

# ----------------------------------------------------------------------------
# Tabelas de origem (Gold ML)
# ----------------------------------------------------------------------------
ML_TRAIN = "credit_risk.gold.ml_train"
ML_TEST = "credit_risk.gold.ml_test"
ML_METADATA = "credit_risk.gold.ml_feature_metadata"

# ----------------------------------------------------------------------------
# Schema de destino (analytics)
# ----------------------------------------------------------------------------
ANALYTICS_SCHEMA = "credit_risk.analytics"
EDA_SUMMARY_TABLE = f"{ANALYTICS_SCHEMA}.eda_summary"
FEATURE_IMPORTANCE_TABLE = f"{ANALYTICS_SCHEMA}.feature_importance_preliminary"
AUDIT_TABLE = f"{ANALYTICS_SCHEMA}.audit_eda"

# ----------------------------------------------------------------------------
# Parâmetros de análise
# ----------------------------------------------------------------------------
SAMPLE_FRACTION = 0.30   # Amostra para correlações e visualizações pesadas
RANDOM_SEED = 42
CORR_THRESHOLD = 0.90   # Threshold para multicolinearidade
TOP_N = 30               # Top N para rankings

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"📊 Sample fraction: {SAMPLE_FRACTION}")
print("✅ Configuração inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 2 — Carregamento e Validação das Tabelas
# ============================================================================
print("=" * 70)
print("CARREGAMENTO E VALIDAÇÃO")
print("=" * 70)

df_train = spark.table(ML_TRAIN)
df_test = spark.table(ML_TEST)
df_metadata = spark.table(ML_METADATA)

train_rc = df_train.count()
test_rc = df_test.count()
meta_rc = df_metadata.count()

train_cols = len(df_train.columns)
test_cols = len(df_test.columns)

print(f"\n   {ML_TRAIN}: {train_rc:,} rows, {train_cols} cols")
print(f"   {ML_TEST}: {test_rc:,} rows, {test_cols} cols")
print(f"   {ML_METADATA}: {meta_rc} rows")

# Validar TARGET
has_target_train = "TARGET" in df_train.columns
has_target_test = "TARGET" in df_test.columns
print(f"\n   TARGET no Train: {'✅' if has_target_train else '❌'}")
print(f"   TARGET no Test: {'✅ (ausente)' if not has_target_test else '❌ (ERRO!)'}")

# SK_ID_CURR
print(f"   SK_ID_CURR Train: {'✅' if 'SK_ID_CURR' in df_train.columns else '❌'}")
print(f"   SK_ID_CURR Test: {'✅' if 'SK_ID_CURR' in df_test.columns else '❌'}")

# Alinhamento de schema
train_feat_set = set(df_train.columns) - {"TARGET"}
test_feat_set = set(df_test.columns)
cols_only_train = train_feat_set - test_feat_set
cols_only_test = test_feat_set - train_feat_set
print(f"\n   Colunas só no Train: {len(cols_only_train)}")
if cols_only_train:
    print(f"      {cols_only_train}")
print(f"   Colunas só no Test: {len(cols_only_test)}")
if cols_only_test:
    print(f"      {cols_only_test}")
schema_aligned = len(cols_only_train) == 0 and len(cols_only_test) == 0
print(f"   Schema alinhado: {'✅' if schema_aligned else '❌'}")

# Type summary
type_counts = {}
for f in df_train.schema.fields:
    dt = f.dataType.simpleString()
    type_counts[dt] = type_counts.get(dt, 0) + 1
print(f"\n   Type summary (Train): {type_counts}")

# Tamanho aproximado (DESCRIBE DETAIL)
try:
    train_detail = spark.sql(f"DESCRIBE DETAIL {ML_TRAIN}").select("sizeInBytes").collect()[0][0]
    test_detail = spark.sql(f"DESCRIBE DETAIL {ML_TEST}").select("sizeInBytes").collect()[0][0]
    print(f"\n   Tamanho Train: {train_detail / (1024**2):.1f} MB")
    print(f"   Tamanho Test: {test_detail / (1024**2):.1f} MB")
except Exception:
    print("\n   Tamanho: não disponível")

print("\n✅ Carregamento e validação concluídos!")

In [0]:
# ============================================================================
# CÉLULA 3 — Visão Geral do Dataset
# ============================================================================
print("=" * 70)
print("VISÃO GERAL DO DATASET")
print("=" * 70)

# Definir features (excluir SK_ID_CURR e TARGET)
ID_COL = "SK_ID_CURR"
TARGET_COL = "TARGET"
EXCLUDE_COLS = {ID_COL, TARGET_COL}
ALL_FEATURES = [c for c in df_train.columns if c not in EXCLUDE_COLS]

# Classificar features por tipo
numeric_features = [f.name for f in df_train.schema.fields
    if f.name in ALL_FEATURES and f.dataType in [T.IntegerType(), T.LongType(), T.DoubleType(), T.FloatType()]]
categorical_features = [f.name for f in df_train.schema.fields
    if f.name in ALL_FEATURES and f.dataType == T.StringType()]

print(f"\n   Total de features de ML: {len(ALL_FEATURES)}")
print(f"   Features numéricas: {len(numeric_features)}")
print(f"   Features categóricas: {len(categorical_features)}")

# Features constantes (approx_count_distinct em única passagem)
distinct_exprs = [F.approx_count_distinct(c).alias(c) for c in ALL_FEATURES]
distinct_row = df_train.agg(*distinct_exprs).collect()[0]
constant_features = [c for c in ALL_FEATURES if (distinct_row[c] or 0) <= 1]
low_variance_features = [c for c in ALL_FEATURES if 1 < (distinct_row[c] or 0) <= 2]

print(f"   Features constantes: {len(constant_features)}")
if constant_features:
    print(f"      {constant_features}")
print(f"   Features baixa variância (≤2 valores): {len(low_variance_features)}")

# NULL total (excluir TARGET)
null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in ALL_FEATURES]
null_row = df_train.agg(*null_exprs).collect()[0]
total_nulls = sum(null_row[c] or 0 for c in ALL_FEATURES)
total_cells = train_rc * len(ALL_FEATURES)
null_pct_total = total_nulls / total_cells * 100 if total_cells > 0 else 0
features_with_null = sum(1 for c in ALL_FEATURES if (null_row[c] or 0) > 0)

print(f"\n   NULL total Train: {total_nulls:,} ({null_pct_total:.2f}%)")
print(f"   Features com NULL: {features_with_null}")

# Tabela resumo
summary_data = [
    ("Clientes Train", f"{train_rc:,}"),
    ("Clientes Test", f"{test_rc:,}"),
    ("Total de Features", str(len(ALL_FEATURES))),
    ("Features Numéricas", str(len(numeric_features))),
    ("Features Categóricas", str(len(categorical_features))),
    ("Features Constantes", str(len(constant_features))),
    ("Features c/ NULL", str(features_with_null)),
    ("NULL Total (%)", f"{null_pct_total:.2f}%"),
]
summary_df = pd.DataFrame(summary_data, columns=["Métrica", "Valor"])
print(f"\n{'─' * 40}")
print("TABELA RESUMO")
print(f"{'─' * 40}")
print(summary_df.to_string(index=False))

print("\n✅ Visão geral concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Análise do TARGET
# ============================================================================
print("=" * 70)
print("ANÁLISE DO TARGET")
print("=" * 70)

# Contagem por classe
class_0 = df_train.filter(F.col(TARGET_COL) == 0).count()
class_1 = df_train.filter(F.col(TARGET_COL) == 1).count()
class_0_pct = class_0 / train_rc * 100
class_1_pct = class_1 / train_rc * 100
imbalance_ratio = class_0 / class_1 if class_1 > 0 else float('inf')

print(f"\n   Classe 0 (adimplente): {class_0:,} ({class_0_pct:.2f}%)")
print(f"   Classe 1 (inadimplente): {class_1:,} ({class_1_pct:.2f}%)")
print(f"   Razão de desbalanceamento: 1:{imbalance_ratio:.2f}")

# Gráfico 1: Distribuição do TARGET (barras)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
classes = ['Classe 0\n(Adimplente)', 'Classe 1\n(Inadimplente)']
counts = [class_0, class_1]
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
        f'{count:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Distribuição do TARGET', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Quantidade')

# Pie chart
axes[1].pie(counts, labels=classes, autopct='%1.2f%%', colors=colors,
    startangle=90, explode=(0, 0.1), shadow=True)
axes[1].set_title('Percentual por Classe', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Conclusões
print(f"\n   Conclusões:")
print(f"   • Dataset altamente desbalanceado (1:{imbalance_ratio:.2f})")
print(f"   • Classe minoritária (inadimplência): {class_1_pct:.2f}%")
print(f"   • Recomenda-se técnicas como class_weight, stratified sampling no Treinamento")
print(f"   • SMOTE NÃO aplicado neste notebook (apenas exploratório)")

print("\n✅ Análise do TARGET concluída!")

In [0]:
# ============================================================================
# CÉLULA 5 — Análise de Missing Values
# ============================================================================
print("=" * 70)
print("ANÁLISE DE MISSING VALUES")
print("=" * 70)

# Calcular null_count e null_percentage para todas as features
null_stats = []
for c in ALL_FEATURES:
    nc = null_row[c] or 0
    np_val = nc / train_rc * 100 if train_rc > 0 else 0
    null_stats.append((c, nc, np_val))

null_df = pd.DataFrame(null_stats, columns=["feature", "null_count", "null_pct"])
null_df = null_df.sort_values("null_count", ascending=False).reset_index(drop=True)

# Top 30 features com mais NULL
print(f"\n   Top 30 Features com MAIS NULL:")
top30_null = null_df.head(30)
print(top30_null.to_string(index=False))

# Distribuição por faixas de NULL
def classify_null_band(pct):
    if pct == 0:
        return "0% (Sem NULL)"
    elif pct <= 5:
        return "0-5%"
    elif pct <= 20:
        return "5-20%"
    elif pct <= 50:
        return "20-50%"
    else:
        return ">50%"

null_df["null_band"] = null_df["null_pct"].apply(classify_null_band)
null_bands = null_df["null_band"].value_counts().sort_index()

print(f"\n   Distribuição de NULL por faixa:")
for band, count in null_bands.items():
    print(f"      {band}: {count} features")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barras: Top 20 features c/ mais NULL
top20_null_plot = null_df[null_df["null_count"] > 0].head(20)
if len(top20_null_plot) > 0:
    axes[0].barh(top20_null_plot["feature"], top20_null_plot["null_pct"], color="#e74c3c")
    axes[0].set_xlabel("NULL (%)")
    axes[0].set_title("Top 20 Features com Mais NULL", fontweight="bold")
    axes[0].invert_yaxis()
else:
    axes[0].text(0.5, 0.5, "Sem NULL", ha="center", va="center", fontsize=16)
    axes[0].set_title("Sem NULL", fontweight="bold")

# Pie: Distribuição por faixa
band_order = ["0% (Sem NULL)", "0-5%", "5-20%", "20-50%", ">50%"]
null_band_counts = null_df["null_band"].value_counts()
band_data = [null_band_counts.get(b, 0) for b in band_order]
band_colors = ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c", "#c0392b"]
axes[1].pie(band_data, labels=band_order, autopct="%1.1f%%", colors=band_colors, startangle=90)
axes[1].set_title("Distribuição de Features por Faixa de NULL", fontweight="bold")

plt.tight_layout()
plt.show()

# Riscos
feats_high_null = null_df[null_df["null_pct"] > 50]
print(f"\n   Riscos identificados:")
print(f"   • {len(feats_high_null)} features com >50% NULL (possível removal)")
if len(feats_high_null) > 0:
    print(f"      {list(feats_high_null['feature'].values[:5])}")
print(f"   • Imputação necessária na etapa de Preparação de Features")

print("\n✅ Análise de Missing Values concluída!")

In [0]:
# ============================================================================
# CÉLULA 6 — Estatísticas das Features Numéricas
# ============================================================================
print("=" * 70)
print("ESTATÍSTICAS DAS FEATURES NUMÉRICAS")
print("=" * 70)

print(f"\n   Total de features numéricas: {len(numeric_features)}")

# Calcular estatísticas em única passagem
stats_exprs = []
for c in numeric_features:
    stats_exprs.extend([
        F.min(c).alias(f"{c}_min"),
        F.max(c).alias(f"{c}_max"),
        F.mean(c).alias(f"{c}_mean"),
        F.expr(f"percentile_approx({c}, 0.5)").alias(f"{c}_median"),
        F.stddev(c).alias(f"{c}_std"),
        F.expr(f"percentile_approx({c}, 0.25)").alias(f"{c}_p25"),
        F.expr(f"percentile_approx({c}, 0.75)").alias(f"{c}_p75"),
        F.skewness(c).alias(f"{c}_skew")
    ])

print(f"   Calculando estatísticas para {len(numeric_features)} features...")
stats_row = df_train.agg(*stats_exprs).collect()[0]

# Organizar resultados
numeric_stats = []
for c in numeric_features:
    numeric_stats.append({
        "feature": c,
        "min": stats_row[f"{c}_min"],
        "max": stats_row[f"{c}_max"],
        "mean": stats_row[f"{c}_mean"],
        "median": stats_row[f"{c}_median"],
        "std": stats_row[f"{c}_std"],
        "p25": stats_row[f"{c}_p25"],
        "p75": stats_row[f"{c}_p75"],
        "skew": stats_row[f"{c}_skew"]
    })

stats_df = pd.DataFrame(numeric_stats)

# Features constantes (min == max)
stats_df["is_constant"] = (stats_df["min"] == stats_df["max"]) | (stats_df["std"].isnull()) | (stats_df["std"] == 0)
constant_numeric = stats_df[stats_df["is_constant"]]
print(f"\n   Features numéricas constantes: {len(constant_numeric)}")
if len(constant_numeric) > 0:
    print(f"      {list(constant_numeric['feature'].values)}")

# Baixa variância (std < 0.01 do mean, exemplo)
stats_df["cv"] = stats_df["std"] / stats_df["mean"].abs()
low_var_numeric = stats_df[(stats_df["cv"] < 0.01) & (~stats_df["is_constant"])]
print(f"   Features baixa variância (CV<0.01): {len(low_var_numeric)}")
if len(low_var_numeric) > 0:
    print(f"      {list(low_var_numeric['feature'].values[:5])}")

# Anomalias potenciais (skewness extrema)
stats_df["skew_abs"] = stats_df["skew"].abs()
high_skew = stats_df[stats_df["skew_abs"] > 5].sort_values("skew_abs", ascending=False)
print(f"   Features com skewness extrema (|skew|>5): {len(high_skew)}")
if len(high_skew) > 0:
    print(f"      Top 5: {list(high_skew['feature'].values[:5])}")

# Top 10 estatísticas
print(f"\n   Top 10 Features (ordenadas por variância):")
top10_stats = stats_df.sort_values("std", ascending=False).head(10)[[
    "feature", "min", "max", "mean", "median", "std", "skew"
]]
print(top10_stats.to_string(index=False))

print("\n✅ Estatísticas das features numéricas concluídas!")

In [0]:
# ============================================================================
# CÉLULA 7 — Distribuições (Histogramas)
# ============================================================================
print("=" * 70)
print("DISTRIBUIÇÕES (HISTOGRAMAS)")
print("=" * 70)

# Selecionar automaticamente top features relevantes
# Usar metadados se disponível, caso contrário usar top por variância
print(f"\n   Selecionando top 20 features relevantes para histogramas...")

# Estrategia: usar top 20 por std (variância)
top20_features = stats_df.sort_values("std", ascending=False).head(20)["feature"].tolist()

# Amostrar para acelerar
print(f"   Amostrando {SAMPLE_FRACTION*100:.0f}% dos dados para visualização...")
df_sample = df_train.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED).select(top20_features).toPandas()

# Gerar histogramas
rows = 5
cols = 4
fig, axes = plt.subplots(rows, cols, figsize=(20, 20))
axes = axes.flatten()

for idx, feat in enumerate(top20_features):
    if idx >= len(axes):
        break
    ax = axes[idx]
    data = df_sample[feat].dropna()
    if len(data) > 0:
        ax.hist(data, bins=30, color="steelblue", edgecolor="black", alpha=0.7)
        ax.set_title(feat, fontweight="bold")
        ax.set_xlabel("Valor")
        ax.set_ylabel("Frequência")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, "Sem dados", ha="center", va="center")
        ax.set_title(feat, fontweight="bold")

plt.suptitle("Distribuições das Top 20 Features", fontsize=18, fontweight="bold", y=1.0)
plt.tight_layout()
plt.show()

# Insights
print(f"\n   Insights:")
print(f"   • Top 20 features selecionadas por variância")
print(f"   • Histogramas gerados a partir de amostra de {SAMPLE_FRACTION*100:.0f}%")
print(f"   • Algumas features mostram distribuições altamente enviesadas")
print(f"   • Transformações (log, sqrt) podem ser necessárias no Pré-Processamento")

print("\n✅ Distribuições concluídas!")

In [0]:
# ============================================================================
# CÉLULA 8 — Análise de Outliers
# ============================================================================
print("=" * 70)
print("ANÁLISE DE OUTLIERS")
print("=" * 70)

print(f"\n   Aplicando método IQR para detectar outliers...")

# Calcular outliers usando IQR
outlier_stats = []
for feat in numeric_features:
    p25 = stats_row[f"{feat}_p25"]
    p75 = stats_row[f"{feat}_p75"]
    
    if p25 is not None and p75 is not None:
        iqr = p75 - p25
        lower_bound = p25 - 1.5 * iqr
        upper_bound = p75 + 1.5 * iqr
        
        # Contar outliers
        outlier_count = df_train.filter(
            (F.col(feat) < lower_bound) | (F.col(feat) > upper_bound)
        ).count()
        
        outlier_pct = outlier_count / train_rc * 100 if train_rc > 0 else 0
        
        # Classificar
        if outlier_pct < 1:
            classification = "Normal"
        elif outlier_pct < 5:
            classification = "Moderado"
        elif outlier_pct < 10:
            classification = "Alto"
        else:
            classification = "Crítico"
        
        outlier_stats.append({
            "feature": feat,
            "outlier_count": outlier_count,
            "outlier_pct": outlier_pct,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "classification": classification
        })

outlier_df = pd.DataFrame(outlier_stats)
outlier_df = outlier_df.sort_values("outlier_pct", ascending=False).reset_index(drop=True)

# Top 20 features com mais outliers
print(f"\n   Top 20 Features com MAIS OUTLIERS:")
top20_outliers = outlier_df.head(20)[[
    "feature", "outlier_count", "outlier_pct", "classification"
]]
print(top20_outliers.to_string(index=False))

# Classificação geral
outlier_class_counts = outlier_df["classification"].value_counts()
print(f"\n   Classificação geral de outliers:")
for cls, count in outlier_class_counts.items():
    print(f"      {cls}: {count} features")

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barras: Top 15 features c/ mais outliers
top15_outliers_plot = outlier_df.head(15)
axes[0].barh(top15_outliers_plot["feature"], top15_outliers_plot["outlier_pct"], color="#e74c3c")
axes[0].set_xlabel("Outlier (%)")
axes[0].set_title("Top 15 Features com Mais Outliers", fontweight="bold")
axes[0].invert_yaxis()

# Pie: Classificação
class_order = ["Normal", "Moderado", "Alto", "Crítico"]
class_counts = [outlier_class_counts.get(c, 0) for c in class_order]
class_colors = ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c"]
axes[1].pie(class_counts, labels=class_order, autopct="%1.1f%%", colors=class_colors, startangle=90)
axes[1].set_title("Distribuição de Features por Classificação de Outliers", fontweight="bold")

plt.tight_layout()
plt.show()

# IMPORTANTE
print(f"\n   IMPORTANTE:")
print(f"   • Outliers NÃO foram removidos (notebook exploratório)")
print(f"   • Features com >10% de outliers podem representar comportamento real")
print(f"   • Revisão detalhada necessária na etapa de Feature Engineering")

print("\n✅ Análise de outliers concluída!")

In [0]:
# ============================================================================
# CÉLULA 9 — Correlação entre Features
# ============================================================================
print("=" * 70)
print("CORRELAÇÃO ENTRE FEATURES")
print("=" * 70)

print(f"\n   Calculando matriz de correlação para {len(numeric_features)} features numéricas...")
print(f"   Usando amostra de {SAMPLE_FRACTION*100:.0f}% para acelerar...")

# Amostrar e converter para pandas
df_corr_sample = df_train.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED).select(numeric_features).toPandas()

# Calcular matriz de correlação
corr_matrix = df_corr_sample.corr()

# Heatmap da matriz de correlação (todas as features)
print(f"   Gerando heatmap...")
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f",
    linewidths=0.1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Matriz de Correlação - Features Numéricas", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Extrair correlações (excluir diagonal e duplicadas)
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        feat1 = corr_matrix.columns[i]
        feat2 = corr_matrix.columns[j]
        corr_val = corr_matrix.iloc[i, j]
        if not np.isnan(corr_val):
            corr_pairs.append((feat1, feat2, corr_val))

corr_pairs_df = pd.DataFrame(corr_pairs, columns=["feature1", "feature2", "correlation"])
corr_pairs_df["abs_corr"] = corr_pairs_df["correlation"].abs()

# Top correlações positivas
print(f"\n   Top 20 Correlações POSITIVAS:")
top_pos = corr_pairs_df[corr_pairs_df["correlation"] > 0].sort_values("correlation", ascending=False).head(20)
print(top_pos[["feature1", "feature2", "correlation"]].to_string(index=False))

# Top correlações negativas
print(f"\n   Top 20 Correlações NEGATIVAS:")
top_neg = corr_pairs_df[corr_pairs_df["correlation"] < 0].sort_values("correlation", ascending=True).head(20)
print(top_neg[["feature1", "feature2", "correlation"]].to_string(index=False))

# Features altamente correlacionadas (threshold)
print(f"\n   Features ALTAMENTE CORRELACIONADAS (|corr| >= {CORR_THRESHOLD}):")
high_corr = corr_pairs_df[corr_pairs_df["abs_corr"] >= CORR_THRESHOLD].sort_values("abs_corr", ascending=False)
print(f"   Total de pares: {len(high_corr)}")
if len(high_corr) > 0:
    print(high_corr[["feature1", "feature2", "correlation"]].head(20).to_string(index=False))
    print(f"\n   Potenciais redundâncias identificadas.")
    print(f"   Recomenda-se remoção de features redundantes na etapa de Feature Selection.")
else:
    print(f"   Nenhum par identificado.")

print(f"\n   IMPORTANTE:")
print(f"   • Features NÃO foram removidas (notebook exploratório)")
print(f"   • Análise baseada em amostra de {SAMPLE_FRACTION*100:.0f}%")

print("\n✅ Análise de correlação entre features concluída!")

In [0]:
# ============================================================================
# CÉLULA 10 — Correlação com TARGET
# ============================================================================
print("=" * 70)
print("CORRELAÇÃO COM TARGET")
print("=" * 70)

print(f"\n   Calculando correlação das features numéricas com TARGET...")

# Usar amostra
df_target_sample = df_train.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED).select(numeric_features + [TARGET_COL]).toPandas()

# Calcular correlações com TARGET
target_corr = df_target_sample[numeric_features].corrwith(df_target_sample[TARGET_COL])
target_corr_df = pd.DataFrame({
    "feature": target_corr.index,
    "correlation": target_corr.values
}).dropna()
target_corr_df["abs_corr"] = target_corr_df["correlation"].abs()
target_corr_df = target_corr_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

# Top correlacoes positivas
print(f"\n   Top 30 Correlações POSITIVAS com TARGET:")
top_pos_target = target_corr_df[target_corr_df["correlation"] > 0].sort_values("correlation", ascending=False).head(30)
print(top_pos_target[["feature", "correlation"]].to_string(index=False))

# Top correlacoes negativas
print(f"\n   Top 30 Correlações NEGATIVAS com TARGET:")
top_neg_target = target_corr_df[target_corr_df["correlation"] < 0].sort_values("correlation", ascending=True).head(30)
print(top_neg_target[["feature", "correlation"]].to_string(index=False))

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Top 20 positivas
if len(top_pos_target) > 0:
    top20_pos = top_pos_target.head(20)
    axes[0].barh(top20_pos["feature"], top20_pos["correlation"], color="#2ecc71")
    axes[0].set_xlabel("Correlação")
    axes[0].set_title("Top 20 Correlações Positivas com TARGET", fontweight="bold")
    axes[0].invert_yaxis()

# Top 20 negativas
if len(top_neg_target) > 0:
    top20_neg = top_neg_target.head(20)
    axes[1].barh(top20_neg["feature"], top20_neg["correlation"], color="#e74c3c")
    axes[1].set_xlabel("Correlação")
    axes[1].set_title("Top 20 Correlações Negativas com TARGET", fontweight="bold")
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\n   IMPORTANTE:")
print(f"   • Análise APENAS exploratória")
print(f"   • NÃO utilizar para excluir features automaticamente")
print(f"   • Features com baixa correlação individual podem ser importantes em combinação")
print(f"   • Análise baseada em amostra de {SAMPLE_FRACTION*100:.0f}%")

print("\n✅ Análise de correlação com TARGET concluída!")

In [0]:
# ============================================================================
# CÉLULA 11 — Análise das Features Categóricas
# ============================================================================
print("=" * 70)
print("ANÁLISE DAS FEATURES CATEGÓRICAS")
print("=" * 70)

print(f"\n   Total de features categóricas: {len(categorical_features)}")

if len(categorical_features) == 0:
    print("\n   Nenhuma feature categórica identificada no dataset.")
else:
    # Analisar cada feature categórica
    categorical_stats = []
    
    for feat in categorical_features:
        # Cardinalidade e distribuição
        freq_df = df_train.groupBy(feat).count().orderBy(F.desc("count"))
        freq_pd = freq_df.toPandas()
        
        cardinality = len(freq_pd)
        total_count = freq_pd["count"].sum()
        
        # Categoria dominante
        if len(freq_pd) > 0:
            dominant_cat = freq_pd.iloc[0][feat]
            dominant_count = freq_pd.iloc[0]["count"]
            dominant_pct = dominant_count / total_count * 100
        else:
            dominant_cat = None
            dominant_count = 0
            dominant_pct = 0
        
        # Categorias raras (<1%)
        rare_cats = freq_pd[freq_pd["count"] / total_count < 0.01]
        rare_count = len(rare_cats)
        
        # Categorias UNKNOWN/MISSING
        unknown_cats = freq_pd[freq_pd[feat].isin(["UNKNOWN", "MISSING", "NA", "NULL", ""])]
        unknown_count = len(unknown_cats)
        
        categorical_stats.append({
            "feature": feat,
            "cardinality": cardinality,
            "dominant_category": dominant_cat,
            "dominant_pct": dominant_pct,
            "rare_categories_count": rare_count,
            "unknown_categories_count": unknown_count
        })
    
    cat_stats_df = pd.DataFrame(categorical_stats)
    cat_stats_df = cat_stats_df.sort_values("cardinality", ascending=False).reset_index(drop=True)
    
    print(f"\n   Resumo das Features Categóricas:")
    print(cat_stats_df.to_string(index=False))
    
    # Features com alta cardinalidade
    high_card = cat_stats_df[cat_stats_df["cardinality"] > 50]
    print(f"\n   Features com alta cardinalidade (>50): {len(high_card)}")
    if len(high_card) > 0:
        print(f"      {list(high_card['feature'].values)}")
    
    # Features dominadas por uma categoria
    high_dom = cat_stats_df[cat_stats_df["dominant_pct"] > 90]
    print(f"   Features dominadas por uma categoria (>90%): {len(high_dom)}")
    if len(high_dom) > 0:
        print(f"      {list(high_dom['feature'].values)}")
    
    print(f"\n   Insights:")
    print(f"   • Features de alta cardinalidade podem requerer encoding especial")
    print(f"   • Features dominadas podem ter baixo poder preditivo")
    print(f"   • Categorias raras podem ser agrupadas em 'OTHER'")

print("\n✅ Análise de features categóricas concluída!")

In [0]:
# ============================================================================
# CÉLULA 12 — Análise TARGET × Features
# ============================================================================
print("=" * 70)
print("ANÁLISE TARGET × FEATURES")
print("=" * 70)

print(f"\n   Comparando distribuições de features por TARGET...")

# Selecionar top features para análise detalhada (top 10 por correlação absoluta)
top_features_for_target = target_corr_df.head(10)["feature"].tolist()

print(f"   Top 10 features selecionadas: {top_features_for_target}")

# NUMÉRICAS: Comparar médias, medianas e percentis por TARGET
print(f"\n   ANÁLISE NUMÉRICAS POR TARGET:")
target_num_stats = []
for feat in top_features_for_target:
    stats_0 = df_train.filter(F.col(TARGET_COL) == 0).agg(
        F.mean(feat).alias("mean_0"),
        F.expr(f"percentile_approx({feat}, 0.5)").alias("median_0")
    ).collect()[0]
    
    stats_1 = df_train.filter(F.col(TARGET_COL) == 1).agg(
        F.mean(feat).alias("mean_1"),
        F.expr(f"percentile_approx({feat}, 0.5)").alias("median_1")
    ).collect()[0]
    
    target_num_stats.append({
        "feature": feat,
        "mean_class_0": stats_0["mean_0"],
        "mean_class_1": stats_1["mean_1"],
        "median_class_0": stats_0["median_0"],
        "median_class_1": stats_1["median_1"]
    })

target_num_df = pd.DataFrame(target_num_stats)
print(target_num_df.to_string(index=False))

# Gráfico: Boxplots comparativos
df_sample_target = df_train.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED).select(
    top_features_for_target + [TARGET_COL]
).toPandas()

fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

for idx, feat in enumerate(top_features_for_target):
    if idx >= len(axes):
        break
    ax = axes[idx]
    data_0 = df_sample_target[df_sample_target[TARGET_COL] == 0][feat].dropna()
    data_1 = df_sample_target[df_sample_target[TARGET_COL] == 1][feat].dropna()
    
    ax.boxplot([data_0, data_1], labels=["Classe 0", "Classe 1"], patch_artist=True,
        boxprops=dict(facecolor="lightblue"), medianprops=dict(color="red", linewidth=2))
    ax.set_title(feat, fontweight="bold")
    ax.set_ylabel("Valor")
    ax.grid(True, alpha=0.3)

plt.suptitle("Distribuições por TARGET (Top 10 Features)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# CATEGÓRICAS: Se houver, calcular taxa de inadimplência por categoria
if len(categorical_features) > 0:
    print(f"\n   ANÁLISE CATEGÓRICAS POR TARGET (primeira feature):")
    first_cat = categorical_features[0]
    cat_target = df_train.groupBy(first_cat, TARGET_COL).count().toPandas()
    cat_pivot = cat_target.pivot(index=first_cat, columns=TARGET_COL, values="count").fillna(0)
    cat_pivot["default_rate"] = cat_pivot[1] / (cat_pivot[0] + cat_pivot[1]) * 100
    cat_pivot = cat_pivot.sort_values("default_rate", ascending=False).head(10)
    print(cat_pivot.to_string())
else:
    print(f"\n   Sem features categóricas para análise TARGET.")

print(f"\n   Insights:")
print(f"   • Diferenças significativas entre classes indicam poder preditivo")
print(f"   • Sobreposição excessiva pode indicar baixa capacidade discriminatória")

print("\n✅ Análise TARGET × Features concluída!")

In [0]:
# ============================================================================
# CÉLULA 13 — Feature Importance Preliminar
# ============================================================================
print("=" * 70)
print("FEATURE IMPORTANCE PRELIMINAR")
print("=" * 70)

print(f"\n   Treinando modelo rápido para estimar importância...")
print(f"   Objetivo: APENAS análise exploratória (sem tuning, sem salvar modelo)")

# Preparar dataset
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Usar apenas features numéricas completas (sem NULL)
numeric_complete = [f for f in numeric_features if (null_row[f] or 0) == 0]
print(f"   Features numéricas completas (sem NULL): {len(numeric_complete)}")

if len(numeric_complete) < 10:
    print(f"   Poucas features completas. Pulando Feature Importance.")
    feature_importance_df = pd.DataFrame()
else:
    # Amostra para acelerar
    df_train_sample = df_train.sample(fraction=SAMPLE_FRACTION, seed=RANDOM_SEED)
    
    # Assembler
    assembler = VectorAssembler(inputCols=numeric_complete, outputCol="features", handleInvalid="skip")
    
    # Random Forest simples
    rf = RandomForestClassifier(
        labelCol=TARGET_COL,
        featuresCol="features",
        numTrees=50,
        maxDepth=5,
        seed=RANDOM_SEED
    )
    
    # Pipeline
    pipeline = Pipeline(stages=[assembler, rf])
    
    print(f"   Treinando Random Forest (50 árvores, depth=5)...")
    model = pipeline.fit(df_train_sample)
    
    # Extrair importâncias
    rf_model = model.stages[-1]
    importances = rf_model.featureImportances.toArray()
    
    feature_importance = list(zip(numeric_complete, importances))
    feature_importance_df = pd.DataFrame(feature_importance, columns=["feature", "importance"])
    feature_importance_df = feature_importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
    
    # Top 30
    print(f"\n   Top 30 Features por Importância:")
    top30_importance = feature_importance_df.head(30)
    print(top30_importance.to_string(index=False))
    
    # Gráfico
    fig, ax = plt.subplots(figsize=(12, 10))
    top20_imp = feature_importance_df.head(20)
    ax.barh(top20_imp["feature"], top20_imp["importance"], color="steelblue")
    ax.set_xlabel("Importância")
    ax.set_title("Top 20 Features por Importância (Random Forest)", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print(f"\n   Insights:")
    print(f"   • Importâncias calculadas via Random Forest simples")
    print(f"   • Modelo NÃO otimizado (apenas exploratório)")
    print(f"   • Features com alta importância são candidatas prioritárias")
    print(f"   • Validação com outros algoritmos recomendada")

print("\n✅ Feature Importance preliminar concluída!")

In [0]:
# ============================================================================
# CÉLULA 14 — Análise de Multicolinearidade
# ============================================================================
print("=" * 70)
print("ANÁLISE DE MULTICOLINEARIDADE")
print("=" * 70)

print(f"\n   Identificando features redundantes e grupos correlacionados...")

# Usar correlações já calculadas
print(f"\n   Features ALTAMENTE CORRELACIONADAS (|corr| >= {CORR_THRESHOLD}):")
if len(high_corr) > 0:
    print(f"   Total de pares: {len(high_corr)}")
    print(high_corr[["feature1", "feature2", "correlation"]].head(20).to_string(index=False))
    
    # Identificar features envolvidas
    features_in_high_corr = set(high_corr["feature1"].tolist() + high_corr["feature2"].tolist())
    print(f"\n   Features envolvidas em alta correlação: {len(features_in_high_corr)}")
    print(f"      {list(features_in_high_corr)[:10]} ...")
    
    # Grupos correlacionados (simplificado: features que aparecem juntas)
    feature_corr_count = {}
    for feat in features_in_high_corr:
        count = len(high_corr[(high_corr["feature1"] == feat) | (high_corr["feature2"] == feat)])
        feature_corr_count[feat] = count
    
    feature_corr_sorted = sorted(feature_corr_count.items(), key=lambda x: x[1], reverse=True)
    print(f"\n   Top 10 features mais correlacionadas com outras:")
    for feat, count in feature_corr_sorted[:10]:
        print(f"      {feat}: {count} pares")
else:
    print(f"   Nenhum par de features com |corr| >= {CORR_THRESHOLD}")

# VIF (simplificado - apenas mencionar)
print(f"\n   VIF (Variance Inflation Factor):")
print(f"   • Cálculo completo de VIF é computacionalmente caro")
print(f"   • Features com |corr| >= {CORR_THRESHOLD} já indicam multicolinearidade")
print(f"   • Recomenda-se Feature Selection para remover redundâncias")

# Conclusão
print(f"\n   Conclusão:")
if len(high_corr) > 0:
    print(f"   • DETECTADA multicolinearidade significativa")
    print(f"   • {len(features_in_high_corr)} features envolvidas")
    print(f"   • Recomenda-se remoção de features redundantes")
    print(f"   • Técnicas: correlação, VIF, PCA, ou feature selection baseada em modelo")
else:
    print(f"   • Multicolinearidade não detectada no threshold {CORR_THRESHOLD}")
    print(f"   • Dataset parece adequado para modelos lineares")

print(f"\n   IMPORTANTE:")
print(f"   • Features NÃO foram removidas (notebook exploratório)")
print(f"   • Decisão de remoção deve ser tomada na etapa de Feature Engineering")

print("\n✅ Análise de multicolinearidade concluída!")

In [0]:
# ============================================================================
# CÉLULA 15 — Análise de Drift Train × Test
# ============================================================================
print("=" * 70)
print("ANÁLISE DE DRIFT TRAIN × TEST")
print("=" * 70)

print(f"\n   Comparando distribuições Train vs Test...")

# Selecionar top features para drift (top 15 por importância se disponível, caso contrário top 15 por variância)
if len(feature_importance_df) > 0:
    drift_features = feature_importance_df.head(15)["feature"].tolist()
    print(f"   Usando top 15 features por importância")
else:
    drift_features = stats_df.sort_values("std", ascending=False).head(15)["feature"].tolist()
    print(f"   Usando top 15 features por variância")

print(f"   Features selecionadas: {drift_features}")

# Calcular médias e medianas
drift_stats = []
for feat in drift_features:
    train_stats = df_train.agg(
        F.mean(feat).alias("mean"),
        F.expr(f"percentile_approx({feat}, 0.5)").alias("median")
    ).collect()[0]
    
    test_stats = df_test.agg(
        F.mean(feat).alias("mean"),
        F.expr(f"percentile_approx({feat}, 0.5)").alias("median")
    ).collect()[0]
    
    mean_train = train_stats["mean"]
    mean_test = test_stats["mean"]
    median_train = train_stats["median"]
    median_test = test_stats["median"]
    
    # Calcular diferença percentual
    if mean_train is not None and mean_test is not None and mean_train != 0:
        mean_diff_pct = abs(mean_test - mean_train) / abs(mean_train) * 100
    else:
        mean_diff_pct = 0
    
    # Classificar drift
    if mean_diff_pct < 5:
        drift_level = "LOW"
    elif mean_diff_pct < 15:
        drift_level = "MEDIUM"
    else:
        drift_level = "HIGH"
    
    drift_stats.append({
        "feature": feat,
        "mean_train": mean_train,
        "mean_test": mean_test,
        "median_train": median_train,
        "median_test": median_test,
        "mean_diff_pct": mean_diff_pct,
        "drift_level": drift_level
    })

drift_df = pd.DataFrame(drift_stats)
drift_df = drift_df.sort_values("mean_diff_pct", ascending=False).reset_index(drop=True)

print(f"\n   Análise de Drift:")
print(drift_df[["feature", "mean_train", "mean_test", "mean_diff_pct", "drift_level"]].to_string(index=False))

# Classificação geral
drift_counts = drift_df["drift_level"].value_counts()
print(f"\n   Classificação geral de drift:")
for level in ["LOW", "MEDIUM", "HIGH"]:
    count = drift_counts.get(level, 0)
    print(f"      {level}: {count} features")

# Gráfico
fig, ax = plt.subplots(figsize=(12, 8))
drift_plot = drift_df.sort_values("mean_diff_pct", ascending=False).head(15)
colors = drift_plot["drift_level"].map({"LOW": "#2ecc71", "MEDIUM": "#f39c12", "HIGH": "#e74c3c"})
ax.barh(drift_plot["feature"], drift_plot["mean_diff_pct"], color=colors)
ax.set_xlabel("Diferença Percentual (%)")
ax.set_title("Drift Train × Test (Top 15 Features)", fontsize=14, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Insights
print(f"\n   Insights:")
if drift_counts.get("HIGH", 0) > 0:
    print(f"   • ATENÇÃO: {drift_counts.get('HIGH', 0)} features com drift HIGH")
    print(f"   • Distribuições Train e Test diferem significativamente")
    print(f"   • Pode impactar generalização do modelo")
else:
    print(f"   • Drift baixo/moderado na maioria das features")
    print(f"   • Distribuições Train e Test são similares")
print(f"   • Validação Cruzada recomendada para medir estabilidade")

print(f"\n   IMPORTANTE:")
print(f"   • Dados NÃO foram alterados (notebook exploratório)")

print("\n✅ Análise de drift concluída!")

In [0]:
# ============================================================================
# CÉLULA 16 — Insights de Negócio
# ============================================================================
print("=" * 70)
print("INSIGHTS DE NEGÓCIO")
print("=" * 70)

print(f"\n   Gerando insights automáticos baseados nos dados...\n")

# ----------------------------------------------------------------------------
# 1. PERFIL DOS CLIENTES INADIMPLENTES
# ----------------------------------------------------------------------------
print("   1️⃣ PERFIL DOS CLIENTES INADIMPLENTES:\n")

# Comparar top features entre classes
if len(target_corr_df) > 0:
    top_corr_positive = target_corr_df[target_corr_df["correlation"] > 0].head(3)
    top_corr_negative = target_corr_df[target_corr_df["correlation"] < 0].head(3)
    
    if len(top_corr_positive) > 0:
        print(f"      • Features associadas à MAIOR inadimplência:")
        for _, row in top_corr_positive.iterrows():
            print(f"         - {row['feature']} (corr={row['correlation']:.3f})")
    
    if len(top_corr_negative) > 0:
        print(f"\n      • Features associadas à MENOR inadimplência:")
        for _, row in top_corr_negative.iterrows():
            print(f"         - {row['feature']} (corr={row['correlation']:.3f})")

# ----------------------------------------------------------------------------
# 2. COMPORTAMENTO FINANCEIRO
# ----------------------------------------------------------------------------
print(f"\n   2️⃣ COMPORTAMENTO FINANCEIRO:\n")

# Buscar features financeiras (assumindo prefixos/sufixos comuns)
financial_keywords = ["AMT_", "CREDIT", "INCOME", "ANNUITY", "GOODS", "_RATIO"]
financial_features = [f for f in numeric_features if any(kw in f.upper() for kw in financial_keywords)]

if len(financial_features) > 0:
    print(f"      • {len(financial_features)} features financeiras identificadas")
    # Calcular estatísticas por TARGET para 3 features financeiras principais
    top_financial = financial_features[:3]
    for feat in top_financial:
        stats_0 = df_train.filter(F.col(TARGET_COL) == 0).agg(F.mean(feat).alias("mean")).collect()[0]["mean"]
        stats_1 = df_train.filter(F.col(TARGET_COL) == 1).agg(F.mean(feat).alias("mean")).collect()[0]["mean"]
        if stats_0 is not None and stats_1 is not None:
            diff_pct = (stats_1 - stats_0) / stats_0 * 100 if stats_0 != 0 else 0
            direction = "maior" if diff_pct > 0 else "menor"
            print(f"         - {feat}: Inadimplentes têm valor {abs(diff_pct):.1f}% {direction}")
else:
    print(f"      • Features financeiras não identificadas automaticamente")

# ----------------------------------------------------------------------------
# 3. HISTÓRICO DE CRÉDITO
# ----------------------------------------------------------------------------
print(f"\n   3️⃣ HISTÓRICO DE CRÉDITO:\n")

# Buscar features de bureau
bureau_keywords = ["BUREAU", "CREDIT", "DEBT", "OVERDUE", "DELAYED"]
bureau_features = [f for f in ALL_FEATURES if any(kw in f.upper() for kw in bureau_keywords)]

if len(bureau_features) > 0:
    print(f"      • {len(bureau_features)} features de histórico de crédito identificadas")
    print(f"      • Histórico de crédito é componente importante do risco")
else:
    print(f"      • Features de histórico de crédito não identificadas automaticamente")

# ----------------------------------------------------------------------------
# 4. UTILIZAÇÃO DE CRÉDITO
# ----------------------------------------------------------------------------
print(f"\n   4️⃣ UTILIZAÇÃO DE CRÉDITO:\n")

# Buscar features de utilização
utilization_keywords = ["BALANCE", "LIMIT", "DRAWINGS", "UTILIZATION"]
utilization_features = [f for f in ALL_FEATURES if any(kw in f.upper() for kw in utilization_keywords)]

if len(utilization_features) > 0:
    print(f"      • {len(utilization_features)} features de utilização de crédito identificadas")
    print(f"      • Padrões de utilização podem indicar stress financeiro")
else:
    print(f"      • Features de utilização não identificadas automaticamente")

# ----------------------------------------------------------------------------
# 5. INDICADORES ASSOCIADOS AO RISCO
# ----------------------------------------------------------------------------
print(f"\n   5️⃣ INDICADORES ASSOCIADOS AO RISCO:\n")

if len(feature_importance_df) > 0:
    top_risk_indicators = feature_importance_df.head(5)
    print(f"      • Top 5 indicadores de risco (por importância):")
    for _, row in top_risk_indicators.iterrows():
        print(f"         - {row['feature']} (imp={row['importance']:.4f})")
else:
    print(f"      • Feature Importance não disponível")

# ----------------------------------------------------------------------------
# CONCLUSÃO GERAL
# ----------------------------------------------------------------------------
print(f"\n   ℹ️ CONCLUSÃO GERAL:\n")
print(f"      • Dataset contém informações ricas sobre comportamento financeiro")
print(f"      • Múltiplas dimensões de risco identificadas")
print(f"      • Combinação de features pode capturar padrões complexos")
print(f"      • Modelo de ML tem potencial para identificar inadimplência")

print("\n✅ Insights de negócio concluídos!")

In [0]:
# ============================================================================
# CÉLULA 17 — Relatório Executivo
# ============================================================================
print("=" * 70)
print("RELATÓRIO EXECUTIVO")
print("=" * 70)

print(f"\n" + "=" * 70)
print("CREDIT RISK INTELLIGENCE PLATFORM")
print("Análise Exploratória de Dados (EDA) - Relatório Executivo")
print("=" * 70)

# ----------------------------------------------------------------------------
# 1. DATASET
# ----------------------------------------------------------------------------
print(f"\n   1. DATASET\n")
print(f"   • Clientes Train: {train_rc:,}")
print(f"   • Clientes Test: {test_rc:,}")
print(f"   • Total de Features: {len(ALL_FEATURES)}")
print(f"   • Features Numéricas: {len(numeric_features)}")
print(f"   • Features Categóricas: {len(categorical_features)}")
print(f"   • Classe 0 (Adimplente): {class_0:,} ({class_0_pct:.2f}%)")
print(f"   • Classe 1 (Inadimplente): {class_1:,} ({class_1_pct:.2f}%)")
print(f"   • Razão de Desbalanceamento: 1:{imbalance_ratio:.2f}")

# ----------------------------------------------------------------------------
# 2. QUALIDADE
# ----------------------------------------------------------------------------
print(f"\n   2. QUALIDADE DOS DADOS\n")
print(f"   • NULL Total: {null_pct_total:.2f}%")
print(f"   • Features com NULL: {features_with_null} de {len(ALL_FEATURES)}")
print(f"   • Features >50% NULL: {len(null_df[null_df['null_pct'] > 50])}")
print(f"   • Features Constantes: {len(constant_features)}")
print(f"   • Outliers (IQR): {len(outlier_df[outlier_df['outlier_pct'] > 10])} features com >10%")

# ----------------------------------------------------------------------------
# 3. CORRELAÇÃO
# ----------------------------------------------------------------------------
print(f"\n   3. CORRELAÇÃO\n")
print(f"   • Correlações analisadas: {len(corr_pairs_df)} pares")
print(f"   • Alta correlação (|r|>={CORR_THRESHOLD}): {len(high_corr)} pares")
if len(high_corr) > 0:
    print(f"   • Features envolvidas: {len(set(high_corr['feature1'].tolist() + high_corr['feature2'].tolist()))}")
if len(target_corr_df) > 0:
    top_target_corr = target_corr_df.head(1)
    print(f"   • Top feature × TARGET: {top_target_corr.iloc[0]['feature']} (r={top_target_corr.iloc[0]['correlation']:.3f})")

# ----------------------------------------------------------------------------
# 4. FEATURE IMPORTANCE
# ----------------------------------------------------------------------------
print(f"\n   4. FEATURE IMPORTANCE (Preliminar)\n")
if len(feature_importance_df) > 0:
    top_feat = feature_importance_df.head(5)
    print(f"   • Top 5 Features:")
    for idx, row in top_feat.iterrows():
        print(f"      {idx+1}. {row['feature']} (imp={row['importance']:.4f})")
else:
    print(f"   • Não calculado")

# ----------------------------------------------------------------------------
# 5. RISCOS
# ----------------------------------------------------------------------------
print(f"\n   5. RISCOS IDENTIFICADOS\n")
risks = []
if imbalance_ratio > 10:
    risks.append("Desbalanceamento severo (1:{:.2f})".format(imbalance_ratio))
if features_with_null > len(ALL_FEATURES) * 0.5:
    risks.append(f"Mais de 50% das features têm NULL")
if len(high_corr) > 10:
    risks.append(f"Multicolinearidade detectada ({len(high_corr)} pares)")
if drift_counts.get("HIGH", 0) > 5:
    risks.append(f"Drift Train×Test em {drift_counts.get('HIGH', 0)} features")
if len(constant_features) > 0:
    risks.append(f"{len(constant_features)} features constantes")

if len(risks) > 0:
    for risk in risks:
        print(f"   • {risk}")
else:
    print(f"   • Nenhum risco crítico identificado")

# ----------------------------------------------------------------------------
# 6. CONCLUSÃO
# ----------------------------------------------------------------------------
print(f"\n   6. CONCLUSÃO\n")

# Decidir se dataset é adequado
critical_issues = 0
if imbalance_ratio > 20:
    critical_issues += 1
if null_pct_total > 30:
    critical_issues += 1
if len(constant_features) > len(ALL_FEATURES) * 0.1:
    critical_issues += 1

if critical_issues == 0:
    dataset_ready = "YES"
    conclusion_msg = "Dataset parece ADEQUADO para treinamento"
else:
    dataset_ready = "YES (com ressalvas)"
    conclusion_msg = "Dataset usável, mas requer atenção em:"

print(f"   • Dataset Ready for Training: {dataset_ready}")
print(f"   • {conclusion_msg}")

if critical_issues > 0:
    if imbalance_ratio > 20:
        print(f"      - Desbalanceamento extremo (usar class_weight/SMOTE)")
    if null_pct_total > 30:
        print(f"      - Alto percentual de NULL (imputação crítica)")
    if len(constant_features) > len(ALL_FEATURES) * 0.1:
        print(f"      - Muitas features constantes (remover)")

print(f"\n   • Próximos passos:")
print(f"      1. Feature Engineering (imputação, transformações)")
print(f"      2. Feature Selection (remover redundantes/irrelevantes)")
print(f"      3. Balanceamento de classes (SMOTE / class_weight)")
print(f"      4. Treinamento de modelos (validação cruzada)")
print(f"      5. Tuning de hiperparâmetros")

print("\n" + "=" * 70)
print("✅ Relatório executivo concluído!")
print("=" * 70)

In [0]:
# ============================================================================
# CÉLULA 18 — Tabelas de Saída (Delta Lake)
# ============================================================================
print("=" * 70)
print("TABELAS DE SAÍDA")
print("=" * 70)

print(f"\n   Criando schema {ANALYTICS_SCHEMA} se não existir...")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ANALYTICS_SCHEMA}")

# ----------------------------------------------------------------------------
# 1. EDA_SUMMARY
# ----------------------------------------------------------------------------
print(f"\n   Criando tabela {EDA_SUMMARY_TABLE}...")

eda_summary_data = [{
    "execution_id": EXECUTION_ID,
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "train_rows": train_rc,
    "test_rows": test_rc,
    "total_features": len(ALL_FEATURES),
    "numeric_features": len(numeric_features),
    "categorical_features": len(categorical_features),
    "class_0_count": class_0,
    "class_1_count": class_1,
    "imbalance_ratio": imbalance_ratio,
    "null_pct_total": null_pct_total,
    "features_with_null": features_with_null,
    "constant_features": len(constant_features),
    "high_corr_pairs": len(high_corr),
    "drift_high_count": drift_counts.get("HIGH", 0),
    "dataset_ready": dataset_ready
}]

eda_summary_spark = spark.createDataFrame([eda_summary_data[0]])
eda_summary_spark.write.mode("append").saveAsTable(EDA_SUMMARY_TABLE)

print(f"   ✅ {EDA_SUMMARY_TABLE} atualizada (append)")

# ----------------------------------------------------------------------------
# 2. FEATURE_IMPORTANCE_PRELIMINARY
# ----------------------------------------------------------------------------
if len(feature_importance_df) > 0:
    print(f"\n   Criando tabela {FEATURE_IMPORTANCE_TABLE}...")
    
    # Adicionar metadados
    feature_importance_df["execution_id"] = EXECUTION_ID
    feature_importance_df["execution_timestamp"] = EXECUTION_TIMESTAMP
    feature_importance_df["pipeline_version"] = PIPELINE_VERSION
    
    # Converter para Spark
    fi_spark = spark.createDataFrame(feature_importance_df)
    fi_spark.write.mode("append").saveAsTable(FEATURE_IMPORTANCE_TABLE)
    
    print(f"   ✅ {FEATURE_IMPORTANCE_TABLE} atualizada (append)")
    print(f"   Registros inseridos: {len(feature_importance_df)}")
else:
    print(f"\n   Feature Importance não disponível. Pulando {FEATURE_IMPORTANCE_TABLE}.")

print("\n✅ Tabelas de saída concluídas!")

In [0]:
# ============================================================================
# CÉLULA 19 — Auditoria
# ============================================================================
print("=" * 70)
print("AUDITORIA")
print("=" * 70)

EXEC_END = datetime.now(timezone.utc)
execution_duration_seconds = (EXEC_END - EXEC_START).total_seconds()

print(f"\n   Criando tabela de auditoria {AUDIT_TABLE}...")

audit_data = {
    "execution_id": EXECUTION_ID,
    "execution_timestamp": EXECUTION_TIMESTAMP,
    "execution_end_timestamp": EXEC_END,
    "execution_duration_seconds": execution_duration_seconds,
    "pipeline_version": PIPELINE_VERSION,
    "notebook_name": NOTEBOOK_NAME,
    "batch_id": BATCH_ID,
    "train_row_count": train_rc,
    "test_row_count": test_rc,
    "feature_count": len(ALL_FEATURES),
    "target_distribution": f"0:{class_0}, 1:{class_1}",
    "null_percentage": null_pct_total,
    "outlier_count": len(outlier_df[outlier_df["outlier_pct"] > 10]),
    "correlated_features": len(high_corr),
    "drift_high_features": drift_counts.get("HIGH", 0),
    "dataset_ready": dataset_ready,
    "status": "SUCCESS"
}

audit_spark = spark.createDataFrame([audit_data])
audit_spark.write.mode("append").saveAsTable(AUDIT_TABLE)

print(f"   ✅ {AUDIT_TABLE} atualizada (append)")
print(f"\n   Detalhes da execução:")
print(f"      Execution ID: {EXECUTION_ID}")
print(f"      Batch ID: {BATCH_ID}")
print(f"      Início: {EXEC_START}")
print(f"      Fim: {EXEC_END}")
print(f"      Duração: {execution_duration_seconds:.2f} segundos")
print(f"      Status: SUCCESS")

print("\n✅ Auditoria concluída!")

In [0]:
# ============================================================================
# CÉLULA 20 — Resumo Final
# ============================================================================
print("=" * 70)
print("RESUMO FINAL")
print("=" * 70)

print(f"\n")

# Tabela final
final_summary = [
    ("Clientes Train", f"{train_rc:,}"),
    ("Clientes Test", f"{test_rc:,}"),
    ("Features", str(len(ALL_FEATURES))),
    ("Features Numéricas", str(len(numeric_features))),
    ("Features Categóricas", str(len(categorical_features))),
    ("Classe 0", f"{class_0:,} ({class_0_pct:.2f}%)"),
    ("Classe 1", f"{class_1:,} ({class_1_pct:.2f}%)"),
    ("Razão de Desbalanceamento", f"1:{imbalance_ratio:.2f}"),
    ("Features com NULL", f"{features_with_null}"),
    ("Features Constantes", str(len(constant_features))),
    ("Features Altamente Correlacionadas", str(len(high_corr))),
    ("Top Feature", feature_importance_df.iloc[0]["feature"] if len(feature_importance_df) > 0 else "N/A"),
    ("Dataset Ready For Training", dataset_ready),
    ("Status", "SUCCESS")
]

final_summary_df = pd.DataFrame(final_summary, columns=["Métrica", "Valor"])

print("\n" + "=" * 70)
print("TABELA RESUMO FINAL")
print("=" * 70)
print(final_summary_df.to_string(index=False))
print("=" * 70)

print(f"\n\n")
print("=" * 70)
print("EDA COMPLETED")
print("=" * 70)
print("READY FOR MODEL TRAINING")
print("=" * 70)
print(f"\n")
print(f"   Execution ID: {EXECUTION_ID}")
print(f"   Batch ID: {BATCH_ID}")
print(f"   Duration: {execution_duration_seconds:.2f}s")
print(f"\n")
print(f"   Tabelas criadas:")
print(f"      • {EDA_SUMMARY_TABLE}")
if len(feature_importance_df) > 0:
    print(f"      • {FEATURE_IMPORTANCE_TABLE}")
print(f"      • {AUDIT_TABLE}")
print(f"\n")
print(f"   Próximos passos: Feature Engineering → Feature Selection → Model Training")
print(f"\n")
print("=" * 70)
print("ALL TASKS COMPLETED SUCCESSFULLY!")
print("=" * 70)